In [0]:
import pyspark.sql.functions as F

In [0]:
class SilverTransformation:
    def __init__(self, catalog_name, bronze_schema_name, bronze_table_name, silver_schema_name, silver_table_name, fillna_columns_datatypes, strictly_drop_columns):
        self.catalog_name = catalog_name
        self.bronze_schema_name = bronze_schema_name
        self.bronze_table_name = bronze_table_name
        self.silver_schema_name = silver_schema_name
        self.silver_table_name = silver_table_name
        self.columns_datatypes = fillna_columns_datatypes
        self.strictly_drop_columns = strictly_drop_columns

    def handle_missing_data(self):
        try:
            print("03: Missing data handling started")
            
            # for strict columns
            self.bronze_data = self.bronze_data.filter(
                F.col(self.strictly_drop_columns[0]).isNotNull() &
                F.col(self.strictly_drop_columns[1]).isNotNull() 
            )

            # for not-strict columns
            for column, datatype in self.columns_datatypes.items():
                if datatype == "string":
                    self.bronze_data = self.bronze_data.fillna({column: "Unknown"})
                elif datatype in ["integer", "double"]:
                    average = self.bronze_data.select(
                        F.avg(F.col(column)).alias("average")
                    ).first()["average"]
                    self.bronze_data = self.bronze_data.fillna({column: average})
                
            print("04: Missing data handling completed")
        except Exception as e:
            print(f"Error: Check Null Blanks Nones | Message - {e}")

    def load_bronze_data(self):
        try:
            bronze_table_path = self.catalog_name + '.' + self.bronze_schema_name + '.' + self.bronze_table_name
            print("01: Bronze table: {bronze_table_path} loading started.")
            self.bronze_data = spark.read.table(bronze_table_path) 
            print("02: Bronze table: {bronze_table_path} Loaded.")

            self.bronze_data.show()
        except Exception as e:
            print(f"Error: Load Bronze Date Function | Message - {e}")

In [0]:
fillna_columns = {
    "transaction_year": "integer",
    "transaction_quarter": "integer",
    "transaction_month": "integer",
    "product_name": "string",
    "product_category": "string",
    "quantity_unit": "string",
    "supplier_name": "string",
    "supplier_country": "string",
    "supplier_reliability_score": "double",
    "refinery_name": "string",
    "destination_city": "string",
    "transportation_mode": "string",
    "ordered_quantity": "double",
    "demand_quantity": "double",
    "available_inventory": "double",
    "unit_price_usd": "double",
    "product_cost_usd": "double",
    "transportation_cost_usd": "double",
    "total_cost_usd": "double",
    "expected_lead_time_days": "integer",
    "actual_lead_time_days": "integer",
    "delay_days": "integer",
    "is_delayed": "integer",
    "is_stockout": "integer",
    "quality_status": "string",
    "quality_score": "double",
    "delivery_status": "string",
}

strictly_drop_columns = ["transaction_id", "transaction_date"]

In [0]:
silver_transformation = SilverTransformation(
    catalog_name="supply_chain",
    bronze_schema_name="bronze",
    bronze_table_name="bronze_supply_chain",
    silver_schema_name="silver",
    silver_table_name="silver_supply_chain",
    fillna_columns_datatypes=fillna_columns,
    strictly_drop_columns=strictly_drop_columns
)

silver_transformation.load_bronze_data()
silver_transformation.handle_missing_data()

In [0]:
import pyspark.sql.functions as F

In [0]:
data = spark.read.table("supply_chain.bronze.bronze_supply_chain")

null_count = dict()
columns = data.columns

columns.remove("transaction_date")
columns.remove("bronze_ingestion_timestamp")

for column in columns:
    print(f"Checking for the column: {column}")
    result = data.select(
        F.sum(
            F.when(
                F.col(column).isNull() |
                (F.trim(F.col(column).cast("string")) == "None")
                , 1
            ).otherwise(0)
        ).alias("null_count")
    ).first()

    null_count[column] = result["null_count"]

print(null_count)


In [0]:
columns = data.columns
print(columns)

In [0]:
data.printSchema()